# Value at Risk Three Ways: Historical, Parametric and Monte Carlo on the DAX

How much could we lose tomorrow? Every trading desk has to answer that question before the market opens, and Value-at-Risk is the industry's standard answer: one number that says “on 99 days out of 100, the loss stays below this line.” The catch is that there are three standard recipes for computing that line — and they disagree exactly when it matters, on the bad days. We compute all three on fifteen years of real DAX data, put every one of them on trial out of sample, and then recompute the historical quantile with Polars and DuckDB to show what the modern data stack changes (answer: the scaling, never the number).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats

plt.rcParams["figure.figsize"] = (10, 5)
SEED = 42

## 1. The number, and the data

Line up fifteen years of daily returns from worst to best. **VaR**at 99% is a marker planted one percent of the way in from the bad end: the loss you do not expect to exceed on 99 days out of 100. Formally it is the (1−α)-quantile of returns, negated — but “the flood line” is the right mental picture. **CVaR** (expected shortfall) answers the question a good risk manager asks next: **and when the water does cross the line, how deep does it get?** It is the average loss given a breach, and we report it alongside every method because it is the number you actually want for limits. Our laboratory is the DAX from 2010-01-04 to 2024-12-30 : 3,805 daily returns spanning the euro crisis, COVID and the 2022 energy shock. The worst day in the sample is 2020-03-12 at `-12.24%` — keep that number in mind while the normal distribution tells you it is impossible.

In [ ]:
px = yf.download("^GDAXI", start="2010-01-01", end="2025-01-01",
                 auto_adjust=True, progress=False)["Close"].squeeze().dropna()
ret = px.pct_change().dropna()
print(f"{len(ret)} daily returns, worst day {ret.idxmin().date()} ({ret.min():.2%})")
ret.plot(lw=0.5, title="DAX daily returns");

## 2. Historical simulation

The first recipe refuses to assume anything. Sort the returns, walk one percent of the way in from the worst end, read off the number — that is the entire method. On the full sample the 99% VaR is `3.42%` and the 99% CVaR — the average of the worst 38 days — is `4.60%`. At 95% the pair is 1.94% / 2.93% .

That honesty cuts both ways. The method treats a sleepy 2017 Tuesday exactly like March 2020, and it has no imagination: it cannot warn you about any loss larger than one it has already lived through. History is its only teacher — a problem, because the worst day of the next fifteen years is under no obligation to have a precedent in the last fifteen.

In [ ]:
def hist_var_cvar(x, alpha=0.99):
    q = np.quantile(x, 1 - alpha)
    return -q, -x[x <= q].mean()

for a in (0.99, 0.95):
    v, c = hist_var_cvar(ret.values, a)
    print(f"historical  {a:.0%}: VaR = {v:.2%}   CVaR = {c:.2%}")

## 3. Parametric: normal, then Student-t

The second recipe trades honesty for a formula: assume returns follow a known distribution, and VaR drops straight out of its quantile function. Assume a normal ( $\mu = 0.039\%$ , $\sigma = 1.228\%$ daily) and the 99% VaR comes out at `2.82%` — roughly 60bp **below**the empirical quantile. That shortfall is the price of the assumption: the bell curve simply does not believe in days as bad as the ones the DAX has actually had. So keep the formula but change the bell. Refit with a Student-t — the normal's heavy-tailed cousin — and maximum likelihood hands you df = `3.22`: violently non-Gaussian tails. The t's 99% VaR of `3.40%` lands almost exactly on the historical number, and its CVaR ( 5.13% ) is fatter still. (One desk convention worth knowing: at the one-day horizon many shops zero out μ entirely — at 0.039% a day it is noise against σ, and estimating it adds error without information. We keep it for completeness.)

Where did the normal lose those 60 basis points? Watch it happen. Below are the three densities over the loss region from −7.5% to −1.5%: the normal (amber) runs out of probability almost immediately — by −3% it has essentially declared such days impossible — while the fitted t tracks the kernel estimate of the real data the whole way down. The 60bp gap in VaR is this picture, integrated.

In [ ]:
mu, sd = ret.mean(), ret.std(ddof=1)
t_df, t_loc, t_scale = stats.t.fit(ret.values)
print(f"fitted Student-t df = {t_df:.2f}")

def normal_var_cvar(alpha):
    z = stats.norm.ppf(1 - alpha)
    return -(mu + sd * z), -mu + sd * stats.norm.pdf(z) / (1 - alpha)

def t_var_cvar(alpha):
    p = 1 - alpha
    x = stats.t.ppf(p, t_df)
    var = -(t_loc + t_scale * x)
    cvar = -t_loc + t_scale * stats.t.pdf(x, t_df) * (t_df + x**2) / ((t_df - 1) * p)
    return var, cvar

for a in (0.99, 0.95):
    nv, nc = normal_var_cvar(a); tv, tc = t_var_cvar(a)
    print(f"{a:.0%}  normal: VaR {nv:.2%} CVaR {nc:.2%}   t: VaR {tv:.2%} CVaR {tc:.2%}")

## 4. Monte Carlo

The third recipe replaces formulas with brute force: simulate 200,000 one-day scenarios from the fitted t (seed 42 ) and read the tail of the simulation as if it were history you simply have not lived yet. On a single linear asset this is a correctness check more than a method — it must reproduce the analytic t to Monte-Carlo error, and it does: `3.39%` vs the analytic 3.40% . So why keep it? Because simulation does not care whether a closed form exists. The moment the book contains options, path dependence, or anything else with no analytic quantile, it is the only recipe still standing.

In [ ]:
rng = np.random.default_rng(SEED)
draws = t_loc + t_scale * rng.standard_t(t_df, 200000)

for a in (0.99, 0.95):
    v, c = hist_var_cvar(draws, a)
    print(f"monte carlo {a:.0%}: VaR = {v:.2%}   CVaR = {c:.2%}")

## 5. The backtest decides

Three recipes, three different answers — so who is right? Here is the beautiful thing about VaR: it is a **falsifiable forecast**. If a 99% VaR is honest, tomorrow's loss should exceed it on about 1% of days — no more, no fewer — and we can simply count. We re-estimate each method on a rolling 250 -day window (t and Monte Carlo refit every 20 days, desk-style), forecast one day ahead — 3,555 out-of-sample forecasts per method — and test the breach count with Kupiec's proportion-of-failures likelihood ratio.

The tell-tale shape: the VaR line jumps **after** each crisis enters the window and relaxes as it leaves. That lag is why breaches cluster — the first breach of the backtest lands on 2011-03-15 , deep in the euro crisis, and the worst runs come in March 2020 before the window has learned what COVID is.

Read it honestly: **every** bar clears the expected line — even the best method understated its own failure rate. The t roughly halves the normal's excess, but Kupiec rejects all four at the 1% level. And Kupiec is the lenient examiner: it only counts breaches, never asking **when** they arrived. One look at the chart shows them arriving in volatility clusters, which is exactly the pattern Christoffersen's (1998) independence test is built to punish and the pattern behind Basel's traffic-light backtest zones. The diagnosis is structural, not a fixable bug: a rolling window only ever looks backwards, so it is late to every regime change by construction. That is not a reason to despair; it is the empirical case for conditional risk models — a GARCH filter rescales the tail to **today's** volatility, and filtered historical simulation is the desk standard for precisely this failure mode.

In [ ]:
WINDOW, REFIT = 250, 20
rv, n = ret.values, len(ret)
z01 = stats.norm.ppf(0.01)

f_hist = ret.rolling(WINDOW).quantile(0.01).shift(1).values
f_norm = (ret.rolling(WINDOW).mean() + ret.rolling(WINDOW).std(ddof=1) * z01).shift(1).values

f_t, f_mc = np.full(n, np.nan), np.full(n, np.nan)
for s in range(WINDOW, n, REFIT):
    dfw, locw, scalew = stats.t.fit(rv[s - WINDOW:s])
    e = min(s + REFIT, n)
    f_t[s:e] = locw + scalew * stats.t.ppf(0.01, dfw)
    f_mc[s:e] = np.quantile(locw + scalew * rng.standard_t(dfw, 200000), 0.01)

def kupiec(x, n, p=0.01):
    phat = x / n
    ll0 = (n - x) * np.log(1 - p) + x * np.log(p)
    ll1 = (n - x) * np.log(1 - phat) + (x * np.log(phat) if x > 0 else 0.0)
    lr = -2 * (ll0 - ll1)
    return lr, stats.chi2.sf(lr, 1)

n_fc = n - WINDOW
for name, f in [("hist", f_hist), ("normal", f_norm), ("t", f_t), ("mc", f_mc)]:
    valid = ~np.isnan(f)
    x = int((rv[valid] < f[valid]).sum())
    lr, p = kupiec(x, n_fc)
    print(f"{name:>6}: {x:>3} breaches / {n_fc} (expected {0.01 * n_fc:.1f})  "
          f"Kupiec LR = {lr:.2f}  p = {p:.4f}")

In [ ]:
plt.plot(ret.index[WINDOW:], rv[WINDOW:], lw=0.4, color="grey", label="daily return")
plt.plot(ret.index[WINDOW:], f_hist[WINDOW:], lw=1.4, label="rolling 99% -VaR (hist)")
breach = rv < f_hist
plt.scatter(ret.index[breach], rv[breach], s=14, color="crimson", zorder=3, label="breach")
plt.legend(); plt.title("DAX daily returns vs rolling 250d historical 99% VaR");
plt.show()

## 6. The same quantile in Polars and DuckDB

Strip the finance away and historical VaR is one line of data engineering: a quantile over a column. That makes it a perfect specimen for a question every quant team eventually asks — does the modern data stack change the answer? **Polars** evaluates a lazy expression pipeline over the CSV; **DuckDB**runs SQL straight against the file. Neither needs the data in pandas, and both stream — the identical two lines still work when “one index, fifteen years” becomes “every book in the firm, tick by tick”.

All three agree to the last printed digit (max absolute difference `0.0e+0`, well inside the 1e−12 tolerance the pipeline asserts). On 3,805 rows the timings are dominated by engine start-up — NumPy answers in a fraction of a millisecond, Polars in a few, DuckDB in tens — so at this size the engines buy you nothing but ergonomics. The crossover comes when the file stops fitting in memory: the pandas version dies, the Polars and DuckDB versions do not change by a character.

In [ ]:
import polars as pl
import duckdb
import time

ret.rename("ret").to_frame().to_csv("dax_returns.csv", index_label="date")

t0 = time.perf_counter()
q_pd = float(np.quantile(rv, 0.01))
print(f"pandas/NumPy   {q_pd:.15f}   {(time.perf_counter() - t0) * 1e3:6.2f} ms")

t0 = time.perf_counter()
q_pl = float(
    pl.scan_csv("dax_returns.csv")
      .select(pl.col("ret").quantile(0.01, interpolation="linear"))
      .collect()
      .item()
)
print(f"polars (lazy)  {q_pl:.15f}   {(time.perf_counter() - t0) * 1e3:6.2f} ms")

t0 = time.perf_counter()
q_db = float(duckdb.sql(
    "SELECT quantile_cont(ret, 0.01) FROM read_csv('dax_returns.csv')"
).fetchone()[0])
print(f"duckdb SQL     {q_db:.15f}   {(time.perf_counter() - t0) * 1e3:6.2f} ms")

# timing note: at 3,805 rows engine startup dominates; the streaming engines
# win once the file no longer fits in memory.
print(f"max |diff| = {max(abs(q_pl - q_pd), abs(q_db - q_pd)):.2e}  (must be <= 1e-12)")

- Jorion, P. Value at Risk: The New Benchmark for Managing Financial Risk. McGraw-Hill.
- Kupiec, P. (1995). Techniques for Verifying the Accuracy of Risk Measurement Models. Journal of Derivatives, 3(2).
- Christoffersen, P. (1998). Evaluating Interval Forecasts. International Economic Review, 39(4), 841–862.
- McNeil, A., Frey, R. & Embrechts, P. Quantitative Risk Management: Concepts, Techniques and Tools. Princeton University Press.
- Companion notebook: `var-three-ways.ipynb` — reproduces every number from raw data (seed 42 ), including the Polars and DuckDB cells.